In [ ]:
train_df = pd.read_csv('/content/dataset_extracted/train_v2.csv')

def simplify_tags(tags):
    if 'artisanal_mining' in tags or 'conventional_mining' in tags: return 'mining'
    if 'road' in tags: return 'logging_road'
    if 'agriculture' in tags or 'cultivation' in tags: return 'agriculture'
    if 'slash_burn' in tags: return 'fire_damage'
    if 'primary' in tags: return 'healthy_forest'
    return 'other'

train_df['label'] = train_df['tags'].apply(simplify_tags)
train_df['image_name'] = train_df['image_name'].apply(lambda x: f"{x}.jpg")

print("Class distribution:")
print(train_df['label'].value_counts())

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    rotation_range=20
)

train_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    directory="/content/dataset_extracted/train-jpg/",
    x_col="image_name",
    y_col="label",
    subset="training",
    batch_size=32,
    seed=42,
    shuffle=True,
    class_mode="categorical",
    target_size=(224, 224) 
)

val_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    directory="/content/dataset_extracted/train-jpg/",
    x_col="image_name",
    y_col="label",
    subset="validation",
    batch_size=32,
    seed=42,
    shuffle=True,
    class_mode="categorical",
    target_size=(224, 224)
)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(len(train_generator.class_indices), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)

model.save('/content/drive/MyDrive/ByteCoders_ForestModel_v2.keras')
print("✅ Multi-class Model trained and saved!")